# L24 · Generalización y transportabilidad

**S24 · Lectura: Cap. 26 · 40–55 min**

## La pregunta

¿El efecto aprendido aquí sirve en otro dominio?

## Qué harás

Comparar poblaciones, soporte y discrepancias antes de transportar.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Un diagnóstico source/target y una decisión de transportar o abstenerse.

> **Idea clave:** Un efecto no viaja solo porque el modelo pueda producir una predicción.

La solución está en `la solución docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
from lab_assumptions import ess_relative
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
rng = setup()

## 1. El diagrama de selección (§26.2.1)
Nodos: $S$ (dominio: 0 = fuente, 1 = destino), $X$ (covariable), $D$ (tratamiento), $Y$ (resultado). Una flecha $S\to V$ marca que **el mecanismo de $V$ difiere** entre dominios. En el caso del Listado 26.1 solo cambia la **distribución de $X$**: $S\to X$, $X\to Y$, $D\to Y$.

La fórmula de reponderación es válida cuando $Y\perp S\mid X,D$ (el mecanismo de $Y$ es el mismo en ambos dominios). Lo verificamos con d-separación.

In [ ]:
# índices: 0=S, 1=X, 2=D, 3=Y
def grafo(aristas):
    A = np.zeros((4, 4)); [A.__setitem__(e, 1) for e in aristas]; return A
G_ok = grafo([(0, 1), (1, 3), (2, 3)])                  # S→X, X→Y, D→Y
G_mal = grafo([(0, 1), (1, 3), (2, 3), (0, 3)])         # además S→Y: el mecanismo de Y cambió
filas = {"Y ⟂ S | {X, D}  (¿es transportable por reponderación?)": [dsep(G_ok, [3], [0], [1, 2]), dsep(G_mal, [3], [0], [1, 2])],
         "Y ⟂ S | {D}      (sin condicionar en X)": [dsep(G_ok, [3], [0], [2]), dsep(G_mal, [3], [0], [2])]}
show(pd.DataFrame(filas, index=["diagrama sin S→Y", "diagrama con S→Y"]).T.replace({True: "sí", False: "no"}))

**Mira el resultado.** En el diagrama sin S→Y, ¿por qué hace falta condicionar en X, y qué pasa si el diagrama tiene además S→Y?

**Respuesta:**

Sin condicionar en $X$, el camino $S\to X\to Y$ está abierto: la distribución de $X$ en cada dominio hace que $Y$ dependa de $S$. Condicionando en $X$ se cierra y el efecto condicional puede llevarse de un dominio a otro. Con $S\to Y$ (el mecanismo difiere) **no hay** forma de cerrarlo con $X$: la reponderación no es válida y el efecto no es transportable con estos datos.



## 2. Los dos dominios (diseño del experimento simulado)
Tomamos las unidades de IHDP-medium (con efecto individual verdadero) y las repartimos en un **dominio fuente** y un **dominio destino** con **selección sobre una covariable continua** $Z$ (la fuente sobre-representa valores altos). En la fuente hay un **ensayo aleatorizado**; en el destino solo observamos covariables, salvo un **grupo aleatorizado de referencia** pequeño para validar.

## Sección de evaluación reservada

La edición pública termina antes de las celdas que requieren `truth`, `y0`, `y1` o `tau` simulados. La evaluación docente se ejecuta en un entorno separado. Continúa la interpretación con datos observados, incertidumbre y supuestos declarados.
